 # Pandapower with UK Power Networks


This tutorial shows some functionalities and studies that can be performed using the power flow capabilities of pandapower. It will demonstrate how to run power flow simulations in pandapower, how to analyse the grid and to investigate different use cases relying on the power flow engine of pandapower.

This tutorial has been created in collaboration with UK Power Networks (UKPN), the Distribution System Operator owning and operating the electricity network across London, the South East and the East of England.

The tutorial will use the real grids associated with the three licensed electricity distribution networks operated by UKPN (LPN, SPN and EPN). It will provide some examples of how pandapower can be used to run investigations and analyses using the open source data released by UKPN. UK Power Networks has provided this grid data as part of their LTDS CIM dataset release. It is a "Shared" dataset that requires special access. To request access:

·       Register and login to the UKPN Open Data Portal [ukpowernetworks.opendatasoft.com]

·       Visit the LTDS CIM [ukpowernetworks.opendatasoft.com] page and complete the Shared Data Request Form [ukpowernetworks.opendatasoft.com]



Once approved, CIM data is published as XML file attachments (one per licence area: EPN, SPN, LPN). You can download the XML files directly from the portal.

Import the pandapower library and the neccessary methods for the conversion as follows:

In [ ]:
import os
import pandapower as pp
from pandapower.converter.cim import from_cim as cim2pp
from pandapower.converter.cim.cim_classes import CimParser
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

## LTDS to pandapower

First, we define the LTDS zip archive, which can be converted to pandapower. If there is no SSH profile available, there is an option to get some P and Q values from Excel. However, please take into account that there are some assumptions made, which limit the applicability of the data and make them unsuitable for all use cases.

Note: If you don't have the Excel files for demand and generation, you can get access to them via the following links:
demand:
https://ukpowernetworks.opendatasoft.com/explore/assets/ltds-table-3a-load-data-observed/view/
generation:
https://ukpowernetworks.opendatasoft.com/explore/assets/ltds-table-5-generation/view/

In [ ]:
# ltds_files is a list containing paths to files needed for the LTDS converter:
ltds_files = [r"path-to-ltds-zip"]
path_excel_demand = r"path-to-excel-input-files"
path_excel_generation = r"path-to-excel-input-files"
excel_column_name = 'Maximum_Demand_24_25_MW'
excel_column_name_pf = 'Maximum_Demand_24_25_PF'
# the Excel data provides the maximum demand / generation. If you want to assume a specific loading,
# choose a scaling_factor between 0.1 and 1.0
scaling_factor = 1.0

cim_parser = CimParser(cgmes_version='ltds')
cim_parser.parse_files(ltds_files).prepare_cim_net().set_cim_data_types()
cim = cim_parser.cim

if os.path.isfile(path_excel_demand):
    excel_df_demand = pd.read_excel(path_excel_demand, sheet_name='Feuil1', skiprows=0)
else:
    excel_df_demand = pd.DataFrame()
if os.path.isfile(path_excel_generation):
    excel_df_generation = pd.read_excel(path_excel_generation, sheet_name='Feuil1', skiprows=0)
else:
    excel_df_generation = pd.DataFrame()
def format_uuid_no_dash(uuid_no_dash: str) -> str | None:
    if uuid_no_dash is None:
        return None
    s = str(uuid_no_dash).lstrip("_")
    if len(s) != 32:
        return uuid_no_dash
    parts = [s[0:8], s[8:12], s[12:16], s[16:20], s[20:32]]
    return '_' + '-'.join(parts)
# prepare the demand data
if not excel_df_demand.empty:
    if 'Season' in excel_df_demand:
        excel_df_demand = excel_df_demand.loc[excel_df_demand['Season'] == 'Winter']
    excel_df_demand = excel_df_demand.rename(columns={'Substation MRID': 'name_excel', excel_column_name: 'p_mw_excel', excel_column_name_pf: 'q_mvar_excel'})
    if 'name_excel' not in excel_df_demand or 'p_mw_excel' not in excel_df_demand or 'q_mvar_excel' not in excel_df_demand:
        # the Excel document is not valid
        excel_df_demand = pd.DataFrame(columns=['name_excel', 'p_mw_excel', 'q_mvar_excel'])
    excel_df_demand = excel_df_demand[['name_excel', 'p_mw_excel', 'q_mvar_excel']].copy()
    excel_df_demand['p_mw_excel'] = excel_df_demand['p_mw_excel'].astype(float)
    excel_df_demand['q_mvar_excel'] = excel_df_demand['q_mvar_excel'].astype(float)
    excel_df_demand['q_mvar_excel'] = ((excel_df_demand['p_mw_excel'] / excel_df_demand['q_mvar_excel'])**2 - excel_df_demand['p_mw_excel']**2)**.5
    excel_df_demand = excel_df_demand.dropna(how='any')
    excel_df_demand_orig = excel_df_demand.copy()
    excel_df_demand = excel_df_demand_orig.groupby('name_excel', as_index=False)['p_mw_excel'].sum()
    excel_df_demand['q_mvar_excel'] = excel_df_demand_orig.groupby('name_excel', as_index=False)['q_mvar_excel'].sum()['q_mvar_excel']
else:
    excel_df_demand = pd.DataFrame(columns=['name_excel', 'p_mw_excel', 'q_mvar_excel'])
# prepare the generation data
if not excel_df_generation.empty:
    if 'Connected_Accepted' in excel_df_generation:
        excel_df_generation = excel_df_generation.loc[excel_df_generation['Connected_Accepted'] == 'Connected']
    excel_df_generation = excel_df_generation.rename(columns={'Substation MRID': 'name_excel', 'InstalledCapacity_MVA': 'p_mw_excel'})
    if 'name_excel' not in excel_df_generation or 'p_mw_excel' not in excel_df_generation:
        # the Excel document is not valid
        excel_df_generation = pd.DataFrame(columns=['name_excel', 'p_mw_excel'])
    excel_df_generation['p_mw_excel'] = excel_df_generation['p_mw_excel'].astype(float)
    excel_df_generation = excel_df_generation.dropna(how='any')
    # note: there might be an issue with the UUID format, this will be fixed with the following line:
    excel_df_generation['name_excel'] = excel_df_generation['name_excel'].apply(format_uuid_no_dash)

    excel_df_generation = excel_df_generation.groupby('name_excel', as_index=False)['p_mw_excel'].sum()
else:
    excel_df_generation = pd.DataFrame(columns=['name_excel', 'p_mw_excel'])

# add the loads to the SSH profile
cim['ssh']['EnergyConsumer'] = pd.concat([cim['ssh']['EnergyConsumer'], cim['eq']['EnergyConsumer'][['rdfId']]], ignore_index=True)
# get the Substation ID
sub = cim['eq']['Terminal'][['ConnectivityNode', 'ConductingEquipment']]
sub = sub.rename(columns={'ConnectivityNode': 'rdfId'})
sub = pd.merge(sub, cim['eq']['ConnectivityNode'][['rdfId', 'ConnectivityNodeContainer']], how='left', on='rdfId')
sub = sub.drop(columns=['rdfId']).drop_duplicates(subset=['ConductingEquipment'])
# adding substations to EnergyConsumer
cim['ssh']['EnergyConsumer']['sub'] = cim['ssh']['EnergyConsumer']['rdfId'].map(
    sub.set_index('ConductingEquipment')['ConnectivityNodeContainer'])
# identify duplications
cim['ssh']['EnergyConsumer']['dups'] = cim['ssh']['EnergyConsumer'].groupby('sub')['sub'].transform('count')
cim['ssh']['EnergyConsumer']['p'] = cim['ssh']['EnergyConsumer']['sub'].map(
    excel_df_demand.set_index('name_excel')['p_mw_excel']) / cim['ssh']['EnergyConsumer']['dups']
cim['ssh']['EnergyConsumer']['p'] = cim['ssh']['EnergyConsumer']['p'].fillna(0.) * scaling_factor
cim['ssh']['EnergyConsumer']['q'] = cim['ssh']['EnergyConsumer']['sub'].map(
    excel_df_demand.set_index('name_excel')['q_mvar_excel']) / cim['ssh']['EnergyConsumer']['dups']
cim['ssh']['EnergyConsumer']['q'] = cim['ssh']['EnergyConsumer']['q'].fillna(0.)
cim['ssh']['EnergyConsumer']['inService'] = cim['ssh']['EnergyConsumer']['inService'].fillna(True)
cim['ssh']['EnergyConsumer'] = cim['ssh']['EnergyConsumer'].drop(columns=['sub', 'dups'])

# add the generation to the SSH profile
for one_asset in ['SynchronousMachine', 'PowerElectronicsConnection']:
    cim['ssh'][one_asset] = pd.concat([cim['ssh'][one_asset], cim['eq'][one_asset][['rdfId']]], ignore_index=True)
    # adding substations to generators
    cim['ssh'][one_asset]['sub'] = cim['ssh'][one_asset]['rdfId'].map(sub.set_index('ConductingEquipment')['ConnectivityNodeContainer'])
    # identify duplications
    cim['ssh'][one_asset]['dups'] = cim['ssh'][one_asset].groupby('sub')['sub'].transform('count')
    cim['ssh'][one_asset]['p'] = cim['ssh'][one_asset]['sub'].map(
        excel_df_generation.set_index('name_excel')['p_mw_excel']) / cim['ssh'][one_asset]['dups']
    cim['ssh'][one_asset]['p'] = cim['ssh'][one_asset]['p'].fillna(0.) * scaling_factor
    cim['ssh'][one_asset]['q'] = cim['ssh'][one_asset]['q'].fillna(0.)
    cim['ssh'][one_asset]['inService'] = cim['ssh'][one_asset]['inService'].fillna(True)
    cim['ssh'][one_asset] = cim['ssh'][one_asset].drop(columns=['sub', 'dups'])

for one_sw in ['Breaker', 'Disconnector', 'Switch', 'LoadBreakSwitch']:
    cim['ssh'][one_sw] = pd.concat([cim['ssh'][one_sw], cim['eq'][one_sw][['rdfId', 'normalOpen']].rename(columns={'normalOpen': 'open'})], ignore_index=True)
    cim['ssh'][one_sw]['inService'] = True

for one_asset in ['ExternalNetworkInjection', 'ConformLoad', 'NonConformLoad', 'StationSupply',
                  'AsynchronousMachine', 'EquivalentInjection']:
    cim['ssh'][one_asset] = pd.concat([cim['ssh'][one_asset], cim['eq'][one_asset][['rdfId']]], ignore_index=True)
    cim['ssh'][one_asset]['p'] = cim['ssh'][one_asset]['p'].fillna(0.) * scaling_factor
    cim['ssh'][one_asset]['q'] = cim['ssh'][one_asset]['q'].fillna(0.)
    cim['ssh'][one_asset]['inService'] = cim['ssh'][one_asset]['inService'].fillna(True)

cim['ssh']['ExternalNetworkInjection']['referencePriority'] = cim['ssh']['ExternalNetworkInjection']['referencePriority'].fillna(1)
cim['ssh']['ExternalNetworkInjection']['controlEnabled'] = cim['ssh']['ExternalNetworkInjection']['controlEnabled'].fillna(True)
cim['ssh']['SynchronousMachine']['referencePriority'] = cim['ssh']['SynchronousMachine']['referencePriority'].fillna(0)
cim['ssh']['SynchronousMachine']['controlEnabled'] = cim['ssh']['SynchronousMachine']['controlEnabled'].fillna(False)
cim['ssh']['EquivalentInjection']['regulationStatus'] = cim['ssh']['EquivalentInjection']['regulationStatus'].fillna(False)

cim['ssh']['EnergySource'] = pd.concat([cim['ssh']['EnergySource'], cim['eq']['EnergySource'][['rdfId']]], ignore_index=True)
cim['ssh']['EnergySource']['activePower'] = cim['ssh']['EnergySource']['activePower'].fillna(0.)
cim['ssh']['EnergySource']['reactivePower'] = cim['ssh']['EnergySource']['reactivePower'].fillna(0.)
cim['ssh']['EnergySource']['inService'] = cim['ssh']['EnergySource']['inService'].fillna(True)

cim['ssh']['StaticVarCompensator'] = pd.concat([cim['ssh']['StaticVarCompensator'], cim['eq']['StaticVarCompensator'][['rdfId']]], ignore_index=True)
cim['ssh']['StaticVarCompensator']['q'] = cim['ssh']['StaticVarCompensator']['q'].fillna(0.)
cim['ssh']['StaticVarCompensator']['inService'] = cim['ssh']['StaticVarCompensator']['inService'].fillna(True)

# add the terminals
cim['ssh']['Terminal'] = pd.concat([cim['ssh']['Terminal'], cim['eq']['Terminal'][['rdfId']]], ignore_index=True)
cim['ssh']['Terminal']['connected'] = cim['ssh']['Terminal']['connected'].fillna(True)
# add the shunts
cim['ssh']['LinearShuntCompensator'] = pd.concat([cim['ssh']['LinearShuntCompensator'], cim['eq']['LinearShuntCompensator'][['rdfId', 'normalSections']].rename(
    columns={'normalSections': 'sections'})], ignore_index=True)
cim['ssh']['LinearShuntCompensator']['controlEnabled'] = cim['ssh']['LinearShuntCompensator']['controlEnabled'].fillna(False)
cim['ssh']['LinearShuntCompensator']['inService'] = cim['ssh']['LinearShuntCompensator']['inService'].fillna(True)
cim['ssh']['NonlinearShuntCompensator'] = pd.concat([cim['ssh']['NonlinearShuntCompensator'], cim['eq']['NonlinearShuntCompensator'][['rdfId']]], ignore_index=True)
cim['ssh']['NonlinearShuntCompensator']['sections'] = cim['ssh']['NonlinearShuntCompensator']['sections'].fillna(0)
cim['ssh']['NonlinearShuntCompensator']['controlEnabled'] = cim['ssh']['NonlinearShuntCompensator']['controlEnabled'].fillna(False)
cim['ssh']['NonlinearShuntCompensator']['inService'] = cim['ssh']['NonlinearShuntCompensator']['inService'].fillna(True)

# add the tap changer steps
cim['ssh']['RatioTapChanger'] = pd.concat([cim['ssh']['RatioTapChanger'], cim['eq']['RatioTapChanger'][['rdfId', 'neutralStep']].rename(
    columns={'neutralStep': 'step'})], ignore_index=True)
cim['ssh']['RatioTapChanger']['controlEnabled'] = cim['ssh']['RatioTapChanger']['controlEnabled'].fillna(False)
# add the TapChangerControls
cim['ssh']['TapChangerControl'] = pd.concat([cim['ssh']['TapChangerControl'], cim['eq']['TapChangerControl'][['rdfId']]], ignore_index=True)
cim['ssh']['TapChangerControl']['discrete'] = cim['ssh']['TapChangerControl']['discrete'].fillna(False)
cim['ssh']['TapChangerControl']['enabled'] = cim['ssh']['TapChangerControl']['enabled'].fillna(False)
cim['eq']['PowerTransformer']['inService'] = True

cim['ssh']['Equipment'] = pd.concat([cim['ssh']['Equipment'], cim['eq']['EquivalentBranch'][['rdfId']]], ignore_index=True)
cim['ssh']['Equipment']['inService'] = cim['ssh']['Equipment']['inService'].fillna(True)
cim['ssh']['Equipment'] = pd.concat([cim['ssh']['Equipment'], cim['eq']['ACLineSegment'][['rdfId']]], ignore_index=True)
cim['ssh']['Equipment']['inService'] = cim['ssh']['Equipment']['inService'].fillna(True)
cim['ssh']['Equipment'] = pd.concat([cim['ssh']['Equipment'], cim['eq']['DCLineSegment'][['rdfId']]], ignore_index=True)
cim['ssh']['Equipment']['inService'] = cim['ssh']['Equipment']['inService'].fillna(True)

# use the from_cim_dict to put in the modified CimParser
net = cim2pp.from_cim_dict(cim_parser=cim_parser, cim_version='LTDS', create_tap_controller=False)

# if there is no slack in the grid, create one
if net.gen.empty and net.ext_grid.empty and not net.sgen.empty:
    slack = net.sgen.loc[net.sgen.in_service].loc[net.sgen.p_mw == net.sgen.p_mw.max()]
    net.sgen = net.sgen.drop(slack.index[0])
    pp.create_gen(net, bus=slack.bus.iloc[0], p_mw=slack.p_mw.iloc[0], slack=True, in_service=True)

print('Conversion successful')

## Get an overview over your grid
Once the network is converted to pandapower, the data can be displayed:

In [ ]:
filename = "LPN EQ SSH_0401_eq.json"   # Give here the name of the json file with the UKPN grid you want to use
net = pp.from_json(filename)
print(net)
# test crash in pipeline - remove!
# test crash in pipeline - remove!
# test crash in pipeline - remove!
# test crash in pipeline - remove!
# test crash in pipeline - remove!
# test crash in pipeline - remove!
# test crash in pipeline - remove!
# test crash in pipeline - remove!
raise ValueError('test crash in pipeline - remove!')
# test crash in pipeline - remove!

## Export your grid
There are different options to export, for example as JSON, Excel or CSV:

In [ ]:
path_to_json = r'../path-to-json.json'
path_to_excel = r'../path-to-excel.xlsx'
path_to_csv = r'path-to-csv'
pp.to_json(net, path_to_json)
pp.to_excel(net, path_to_excel)
if os.path.isdir(path_to_csv):
    # for CSV, there is no pandapower method available
    for one_type in ['bus', 'line', 'impedance', 'trafo', 'trafo3w', 'load', 'sgen', 'gen']:
        net[one_type].to_csv(path_to_csv+'\\'+one_type+'.csv', sep=',')
else:
    print("Please provide a valid path for exporting the CSV data.")

## Display the nodes

In [ ]:
print(f"Overview over the nodes: {net.bus.describe()}")
print(f"The nodes: {net.bus.loc[:100]}")

## Display the lines

In [ ]:
print(f"Overview over the lines: {net.line.describe()}")
print(f"The lines: {net.line.loc[:100]}")

## Display the transformers

In [ ]:
# two winding transformers
print(f"Overview over the 2W transformers: {net.trafo.describe()}")
print(f"The 2W transformers: {net.trafo.loc[:100]}")
# three winding transformers
print(f"Overview over the 3W transformers: {net.trafo3w.describe()}")
print(f"The 3W transformers: {net.trafo3w.loc[:100]}")

## Display the loads

In [ ]:
print(f"Overview over the loads: {net.load.describe()}")
print(f"The loads: {net.load.loc[:100]}")

## Display the generation

In [ ]:
print(f"Overview over the PQ generators: {net.sgen.describe()}")
print(f"The PQ generators: {net.sgen.loc[:100]}")

print(f"Overview over the PV generators: {net.gen.describe()}")
print(f"The PV generators: {net.gen.loc[:100]}")

## Get only HV elements from the grid
In pandapower we are using pandas DataFrames, you can create your queries like you wish. Here is an example to get HV (110kV) elements from your grid.

In [ ]:
print("the HV nodes first")
print(net.bus.loc[(net.bus.vn_kv > 100) & (net.bus.vn_kv < 150)])

In [ ]:
print("now the HV lines")
net.line['vn_kv_bus'] = net.line.from_bus.map(net.bus.vn_kv)
print(net.line.loc[(net.line.vn_kv_bus > 100) & (net.line.vn_kv_bus < 150)])

In [ ]:
print("now the HV 2W trafos")
print(net.trafo.loc[(net.trafo.vn_hv_kv > 100) & (net.trafo.vn_hv_kv < 150)])

In [ ]:
print("now the HV 3W trafos")
print(net.trafo3w.loc[(net.trafo3w.vn_hv_kv > 100) & (net.trafo3w.vn_hv_kv < 150)])

In [ ]:
print("now the loads")
net.load['vn_kv_bus'] = net.load.bus.map(net.bus.vn_kv)
print(net.load.loc[(net.load.vn_kv_bus > 100) & (net.load.vn_kv_bus < 150)])

In [ ]:
print("now the PQ generators")
net.sgen['vn_kv_bus'] = net.sgen.bus.map(net.bus.vn_kv)
print(net.sgen.loc[(net.sgen.vn_kv_bus > 100) & (net.sgen.vn_kv_bus < 150)])

In [ ]:
print("now the PV generators")
net.gen['vn_kv_bus'] = net.gen.bus.map(net.bus.vn_kv)
print(net.gen.loc[(net.gen.vn_kv_bus > 100) & (net.gen.vn_kv_bus < 150)])

# Run a power flow

#### Workarounds for power flow execution
The following blocks of code provide some functions to apply some workarounds necessary to run successfully the power flow on the UK Power Networks grids.
These workarounds include, for example, the creation of external grids (*slack buses* in the power flow terminology) or the replacement of zero impedance components with switches. 

In [ ]:
# Function to replace components with very small impedance with switches.
from pandapower.toolbox import create_replacement_switch_for_branch

def _replace_zero_impedance_components(net):
    min_ohm = 0.001
    to_replace = (np.abs(net.line.x_ohm_per_km * net.line.length_km) <= min_ohm) & net.line.in_service

    if np.any(to_replace):
        print(f"replaced {sum(to_replace)} lines with switches")

    for i in net.line.loc[to_replace].index.values:
        create_replacement_switch_for_branch(net, "line", i)
        net.line.at[i, "in_service"] = False

    xward = net.xward.loc[(np.abs(net.xward.x_ohm) <= min_ohm) & net.xward.in_service].index.values
    if len(xward) > 0:
        pp.replace_xward_by_ward(net, index=xward, drop=False)
        print(f"replaced {len(xward)} xwards with wards")

    zb_f_ohm = np.square(net.bus.loc[net.impedance.from_bus.values, "vn_kv"].values) / net.impedance.sn_mva
    zb_t_ohm = np.square(net.bus.loc[net.impedance.to_bus.values, "vn_kv"].values) / net.impedance.sn_mva
    impedance = ((np.abs(net.impedance.xft_pu) <= min_ohm / zb_f_ohm) |
                (np.abs(net.impedance.xtf_pu) <= min_ohm / zb_t_ohm)) & net.impedance.in_service

    if any(impedance):
        print(f"replaced {sum(impedance)} impedance elements with switches")

    for i in net.impedance.loc[impedance].index.values:
        pp.create_replacement_switch_for_branch(net, "impedance", i)
        net.impedance.at[i, "in_service"] = False

In [ ]:
# Function to apply the needed workarounds
def apply_workarounds(net, license_area):
    net.impedance.drop(net.impedance.index, inplace=True)
    _replace_zero_impedance_components(net)
    net.line["c_nf_per_km"] *= 0.1
    net.load["p_mw"] *= 0.1

    if license_area == "LPN":
        pp.create_ext_grid(net,bus=10711,vm_pu=1)
        pp.create_ext_grid(net,bus=10699,vm_pu=1)
        pp.create_ext_grid(net,bus=10674,vm_pu=1)
        pp.create_ext_grid(net,bus=10738,vm_pu=1)
        pp.create_ext_grid(net,bus=10673,vm_pu=1)
    elif license_area == "SPN":
        pp.create_ext_grid(net,bus=4899,vm_pu=1)
        pp.create_ext_grid(net,bus=4879,vm_pu=1)
        pp.create_ext_grid(net,bus=4903,vm_pu=1)
        pp.create_ext_grid(net,bus=4920,vm_pu=1)
        pp.create_ext_grid(net,bus=4916,vm_pu=1)
        pp.create_ext_grid(net,bus=4925,vm_pu=1)
        pp.create_ext_grid(net,bus=4878,vm_pu=1)
    elif license_area == "EPN":
        pp.create_ext_grid(net,bus=9906,vm_pu=1)
        pp.create_ext_grid(net,bus=9918,vm_pu=1)
        pp.create_ext_grid(net,bus=9900,vm_pu=1)
        pp.create_ext_grid(net,bus=9910,vm_pu=1)
        pp.create_ext_grid(net,bus=9878,vm_pu=1)
    else:
        raise ValueError("Sorry, this license area does not exist in UK Power Networks. Allowed areas are LPN, SPN and EPN.")

    return net


In [ ]:
# Apply the workarounds on the selected grid
license_area = "LPN"  # Provide here the name of the considered license area. It should be "LPN", "SPN", or "EPN".
if net.bus.index.size > 1:
    net = apply_workarounds(net, license_area)

## Run a power flow study
One of the easiest tasks that can be done with pandapower is to run a power flow. 
This allows analysing the voltage conditions in the grid and the powers/currents flowing through the different lines and components of the network, considering the load and generation available as input. 

Through a power flow calculation it is possible to make a contingency analysis, namely to assess if the operating conditions of the grid are within the allowed boundaries.

In this section, you will see: 
- How to run a power flow and visualize the results
- How to filter the power flow results
- How to identify possible contingencies (overloading or voltage violations)



In [ ]:
# Run a power flow
if len(net.bus) == 0:
    bus = pp.create_bus(net, vn_kv=132)
    pp.create_ext_grid(net, bus=bus)
pp.runpp(net, max_iteration=50)

## Visualize the power flow results
In the bus results table you will find the resulting bus voltage and power consumption / injection at each bus

In [ ]:
# Visualize bus results
display(net.res_bus)
display("Maximum voltage magnitude in the grid (per unit): " + "{:.4f}".format(np.nanmax(net.res_bus.vm_pu)))
display("Minimum voltage magnitude in the grid (per unit): " + "{:.4f}".format(np.nanmin(net.res_bus.vm_pu)))

Some of the bus results may have NaN. This happens for those buses that are disconnected from the main grid.

In [ ]:
# Visualize number of connected buses
num_disconnected_buses = np.sum(np.isnan(net.res_bus.vm_pu))
num_connected_buses = np.sum(~np.isnan(net.res_bus.vm_pu))
total_num_buses = len(net.bus)
percentage_connected_buses = 100 * num_connected_buses / total_num_buses
display("Percentage of connected buses: " + "{:.2f}".format(percentage_connected_buses) + "%")

In the line and transformer result tables you can see, among others, the level of power flowing through these components.

In [ ]:
# Visualize line results
display(net.res_line)
display("Maximum active power in the lines: " + "{:.2f}".format(net.res_line.loc[net.res_line.p_from_mw.notna(), 'p_from_mw'].max()) + " MW")

In [ ]:
# Visualize transformer results
display(net.res_trafo)
display("Maximum active power in the transformers: " + "{:.2f}".format(net.res_trafo.loc[net.res_trafo.p_hv_mw.notna(), 'p_hv_mw'].max()) + " MW")

You can visualize the results for a specific element

In [ ]:
# Visualize bus results at bus 45
bus_idx = 45
if bus_idx in net.res_bus.index:
    print(net.res_bus.loc[bus_idx])
else:
    print("The given bus does not exist")

In [ ]:
# Visualize results for transformer 15
if 15 in net.res_trafo.index:
    print(net.res_trafo.loc[15])
else:
    print("The given transformer does not exist")

## Sort the results 
You can easily sort the results using the *sort_values* function


In [ ]:
# Sort bus results from buses with the smallest voltage
net.res_bus.sort_values("vm_pu").head(20)

In [ ]:
# Sort line results from lines with highest active power flow
net.res_line.sort_values("p_from_mw", ascending=False).head(20)

## Filter the results 
You can filter the results as you like, selecting only specific types or clusters of elements, or specific columns of the tables

In [ ]:
# Visualize bus results only for buses at 132 kV
net.res_bus[net.bus.vn_kv==132]

In [ ]:
# Visualize transformer results only for 132 kV/33 kV  transformers 
net.res_trafo[(net.trafo.vn_hv_kv==132) & (net.trafo.vn_lv_kv==33)]

In [ ]:
# Visualize only active and reactive powers of the lines
net.res_line[["p_from_mw", "q_from_mvar", "p_to_mw", "q_to_mvar"]]

## Detect contingencies
You can easily identify possible voltage contingencies in the grid, namely voltage values beyond the allowed thresholds. 

In [ ]:
# Check possible voltage violations
# Define voltage boundaries
lower_v_threshold = 0.90   # Define the lower boundary of the voltage magnitude (in per unit)
upper_v_threshold = 1.10   # Define the upper boundary of the voltage magnitude (in per unit)

# Check for overvoltages
if np.any(net.res_bus.vm_pu > upper_v_threshold):
    display("Overvoltages are present in the grid. Maximum voltage is: " + "{:.4f}".format(np.nanmax(net.res_bus.vm_pu)) + " p.u.")
    buses_with_overvoltage = net.bus.index[net.res_bus.vm_pu>upper_v_threshold]
else: 
    display("No overvoltages are present in the grid")

# Check for undervoltages
if np.any(net.res_bus.vm_pu < lower_v_threshold):
    display("Undervoltages are present in the grid. Minimum voltage is: " + "{:.4f}".format(np.nanmin(net.res_bus.vm_pu)) + " p.u.")
    buses_with_undervoltage = net.bus.index[net.res_bus.vm_pu<lower_v_threshold]
else: 
    display("No undervoltages are present in the grid")



You can easily check if any overloading exists in the grid (**NB**: the possibility of verifying overloadings depends on the availability of rated values for lines and/or transformers).

In [ ]:
# Check overloadings for transformers

overloading_factor = 1  # You can define an overloading factor if you desire to check overloadings for values different from 100%
if np.any(net.res_trafo.loading_percent > 100*overloading_factor):
    display("Overloading present in the grid transformers. Maximum loading is: " + "{:.2f}".format(np.nanmax(net.res_trafo.loading_percent)) + "%")
else: 
    display("No overloading is present in the grid transformers")

In [ ]:
# Visualize transformer loading (results sorted by the largest loading)
net.res_trafo[["loading_percent"]].sort_values("loading_percent", ascending=False)

# Hosting capacity: impact of new load or generation connections

Hosting capacity studies are a common use case that can be addressed leveraging the pandapower power flow libraries. The goal is to understand how much load or generation can be connected to a bus, before exceeding the allowed boundaries (voltage boundaries or overloading of the grid components).

In this section you will see:
- How to add new loads or generators to the grid
- How to discover the maximum load or generation that can be added at a bus before exceeding the operational boundaries (i.e., voltage or overloading limits)

## Add a new load 
A new load can be easily created with the *create_load" function of pandapower. It requires defining the bus to which the load will be connected and its active and reactive power.

In [ ]:
# Create a new load at the desired bus
load_bus = 3403

if load_bus not in net.bus.index:
    # if the bus not exists, it needs to be created first
    pp.create_bus(net, vn_kv=132, index=load_bus)
load_p = 0.2
load_q = 0.1
pp.create_load(net, bus=load_bus, p_mw=load_p, q_mvar=load_q)
net.load.tail(1)

## Add a new generator
A new static generator can be easily created with the *create_sgen" function of pandapower. It requires defining the bus to which the static generator will be connected and its active and reactive power.

In [ ]:
# Create a new sgen at the desired bus
sgen_bus = 3403

if sgen_bus not in net.bus.index:
    # if the bus not exists, it needs to be created first
    pp.create_bus(net, vn_kv=132, index=sgen_bus)
sgen_p = 0.3
sgen_q = 0
pp.create_sgen(net, bus=sgen_bus, p_mw=sgen_p, q_mvar=sgen_q)
net.sgen.tail(1)

## Run a hosting capacity analysis
It is possible to run a hosting capacity study and understand how much load or generation can be connected to a particular node, by incrementing continuously the power (of the load or generator) till when the boundaries of interest are not exceeded.

In this example, for simplicity, we will investigate how much load can be added to the desired bus before exceeding the loading capacity of the grid transformers.

In [ ]:
if license_area == "LPN":
    hosting_bus = 8
elif license_area == "SPN":
    hosting_bus = 36
elif license_area == "EPN":
    hosting_bus = 20
else:
    hosting_bus = 0             # bus selected for the analysis
            # bus selected for the analysis
incremental_p_mw = 1        # incremental value of power
if hosting_bus not in net.bus.index:
    print("The given bus does not exist")
    hosting_bus = net.bus.index[0]   # replace the bus with the first bus in the grid
load_index = pp.create_load(net, bus=hosting_bus, p_mw=0, q_mvar=0)
within_hosting_limit = True    # boolean telling if we are still within inside the allowed boundary

# Hosting capacity logic
while within_hosting_limit:
    net.load.loc[load_index, "p_mw"] += incremental_p_mw
    pp.runpp(net, max_iteration=50)
    if np.any(net.res_trafo.loading_percent > 100) or net.trafo.index.size == 0:
        within_hosting_limit = False
        net.load.loc[load_index, "p_mw"] -= incremental_p_mw

# Visualize maximum load that can be connected at the selected bus
display("Maximum load that can be connected at bus " + str(load_index) + " is " + str(net.load.loc[load_index, "p_mw"]) + " MW")